<a href="https://colab.research.google.com/github/twahanur/Skin-cancer-Classification-/blob/main/SkinCancerClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Kaggle API setup
!pip install -q kaggle


from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Dataset download + unzip (HAM10000)
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000
!unzip -q skin-cancer-mnist-ham10000.zip -d ham10000

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
License(s): CC-BY-NC-SA-4.0
100% 5.20G/5.20G [00:30<00:00, 180MB/s]



In [2]:
!pip install -q timm scikit-learn pandas pillow matplotlib seaborn torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121

import os
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
import timm

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = Path("ham10000")
print("Using device:", device)
print("Data directory:", DATA_DIR.resolve())

Using device: cuda
Data directory: /content/ham10000


In [3]:
# Metadata load + path checks + stratified train/val/test split
df = pd.read_csv(DATA_DIR/'HAM10000_metadata.csv')

class_map = {
    'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3,
    'mel': 4, 'nv': 5, 'vasc': 6
}
idx_to_class = {v: k for k, v in class_map.items()}
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]

def resolve_image_path(image_id: str) -> str:
    p1 = DATA_DIR / 'HAM10000_images_part_1' / f'{image_id}.jpg'
    p2 = DATA_DIR / 'HAM10000_images_part_2' / f'{image_id}.jpg'
    return str(p1 if p1.exists() else p2)

df['label'] = df['dx'].map(class_map)
df['image_path'] = df['image_id'].apply(resolve_image_path)
df['is_file_ok'] = df['image_path'].apply(os.path.exists)

missing_files = (~df['is_file_ok']).sum()
if missing_files > 0:
    raise FileNotFoundError(f"{missing_files} image files are missing. Please re-check dataset unzip path.")

print("Total samples:", len(df))
print("Class distribution (full data):")
print(df['dx'].value_counts())

# Split 1: train_val + test (80 / 20)
train_val_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=SEED
)

# Split 2: train + val (75 / 25 from train_val => 60 / 20 / 20 overall)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.25, stratify=train_val_df['label'], random_state=SEED
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Class-balanced weights are computed only from train split
beta = 0.999
n_samples = train_df['label'].value_counts().sort_index().values
effective_num = 1.0 - np.power(beta, n_samples)
weights = (1.0 - beta) / effective_num
class_weights = torch.tensor(weights / weights.sum(), dtype=torch.float32).to(device)
print("Class weights:", class_weights)

Total samples: 10015
Class distribution (full data):
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
Train: 6009 | Val: 2003 | Test: 2003
Class weights: tensor([0.1339, 0.0903, 0.0496, 0.3590, 0.0491, 0.0244, 0.2937],
       device='cuda:0')


In [4]:
class HAMDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_path = self.df.loc[idx, 'image_path']
        image = Image.open(img_path).convert('RGB')
        label = int(self.df.loc[idx, 'label'])

        if self.transform is not None:
            image = self.transform(image)
        return image, label

# Train augmentation for better generalization
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Deterministic eval transform for val/test
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = HAMDataset(train_df, train_transform)
val_dataset = HAMDataset(val_df, eval_transform)
test_dataset = HAMDataset(test_df, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Dataloaders ready -> Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

Dataloaders ready -> Train: 6009, Val: 2003, Test: 2003


In [ ]:
# ConvNeXt-Base + Swin-Base fine-tuning (train -> val)
model_cnn = timm.create_model('convnext_base', pretrained=True, num_classes=7).to(device)
model_swin = timm.create_model('swin_base_patch4_window7_224', pretrained=True, num_classes=7).to(device)

def evaluate_model(model, loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            logits = model(images)
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.cpu().numpy())
            targets.extend(labels.cpu().numpy())
    return accuracy_score(targets, preds), np.array(preds), np.array(targets)

def fine_tune_model(model, model_name, epochs=5):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.2, patience=1)

    best_val_acc = 0.0
    history = []
    print(f"\nStarting fine-tuning: {model_name} on {device}")

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        progress = tqdm(train_loader, desc=f"{model_name} Epoch {epoch + 1}/{epochs}", leave=False)

        for images, labels in progress:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            progress.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_loss / max(len(train_loader), 1)
        val_acc, _, _ = evaluate_model(model, val_loader)
        scheduler.step(val_acc)

        history.append({
            'epoch': epoch + 1,
            'train_loss': avg_train_loss,
            'val_acc': val_acc
        })
        print(f"{model_name} | Epoch {epoch + 1}/{epochs} -> train_loss: {avg_train_loss:.4f}, val_acc: {val_acc * 100:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"best_{model_name}.pth")
            print(f"Saved best checkpoint: best_{model_name}.pth")

    print(f"Best {model_name} validation accuracy: {best_val_acc * 100:.2f}%")
    return history

history_cnn = fine_tune_model(model_cnn, "ConvNeXt-Base", epochs=5)
history_swin = fine_tune_model(model_swin, "Swin-Base", epochs=5)

model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]


Starting fine-tuning: ConvNeXt-Base on cuda


ConvNeXt-Base Epoch 1/5:   0%|          | 0/188 [00:00<?, ?it/s]

ConvNeXt-Base | Epoch 1/5 -> train_loss: 1.2825, val_acc: 74.69%
Saved best checkpoint: best_ConvNeXt-Base.pth


...

ConvNeXt-Base Epoch 3/5:   0%|          | 0/188 [00:00<?, ?it/s]

ConvNeXt-Base | Epoch 3/5 -> train_loss: 0.5358, val_acc: 78.13%


ConvNeXt-Base Epoch 4/5:   0%|          | 0/188 [00:00<?, ?it/s]

ConvNeXt-Base | Epoch 4/5 -> train_loss: 0.4320, val_acc: 83.92%
Saved best checkpoint: best_ConvNeXt-Base.pth


ConvNeXt-Base Epoch 5/5:   0%|          | 0/188 [00:00<?, ?it/s]

### 1. ক্লাস ডিস্ট্রিবিউশন (Class Distribution)

প্রথমে, আমরা `train_df` এবং `test_df` উভয় ডেটাসেটে লেবেলের বন্টন ভিজ্যুয়ালাইজ করব।

In [ ]:
# Feature extraction (no leakage): fit on train, tune on val, report on test
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Reload best checkpoints before feature extraction
model_cnn = timm.create_model('convnext_base', pretrained=False, num_classes=7).to(device)
model_swin = timm.create_model('swin_base_patch4_window7_224', pretrained=False, num_classes=7).to(device)
model_cnn.load_state_dict(torch.load('best_ConvNeXt-Base.pth', map_location=device), strict=True)
model_swin.load_state_dict(torch.load('best_Swin-Base.pth', map_location=device), strict=True)

model_cnn.eval()
model_swin.eval()

def pool_features(feat):
    if feat.dim() == 4:
        return feat.mean(dim=[2, 3])
    if feat.dim() == 3:
        return feat.mean(dim=1)
    if feat.dim() == 2:
        return feat
    raise ValueError(f"Unexpected feature tensor shape: {feat.shape}")

def extract_fused_features(cnn_model, swin_model, loader):
    all_features, all_labels = [], []
    with torch.no_grad():
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device, non_blocking=True)
            cnn_feat = pool_features(cnn_model.forward_features(images))
            swin_feat = pool_features(swin_model.forward_features(images))
            fused = torch.cat([cnn_feat, swin_feat], dim=1)
            all_features.append(fused.cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_features), np.concatenate(all_labels)

X_train_raw, y_train = extract_fused_features(model_cnn, model_swin, train_loader)
X_val_raw, y_val = extract_fused_features(model_cnn, model_swin, val_loader)
X_test_raw, y_test = extract_fused_features(model_cnn, model_swin, test_loader)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test = scaler.transform(X_test_raw)

classifiers = {
    'SVM': SVC(kernel='rbf', C=5.0, probability=True, class_weight='balanced', random_state=SEED),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'LR': LogisticRegression(max_iter=2000, class_weight='balanced', random_state=SEED)
}

eval_results = {}
for name, clf in classifiers.items():
    clf.fit(X_train, y_train)
    val_pred = clf.predict(X_val)
    test_pred = clf.predict(X_test)
    val_acc = accuracy_score(y_val, val_pred)
    test_acc = accuracy_score(y_test, test_pred)
    test_proba = clf.predict_proba(X_test) if hasattr(clf, 'predict_proba') else None

    eval_results[name] = {
        'model': clf,
        'val_pred': val_pred,
        'test_pred': test_pred,
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_proba': test_proba
    }

    print(f"\n{name} -> Val Acc: {val_acc * 100:.2f}% | Test Acc: {test_acc * 100:.2f}%")
    print(classification_report(y_test, test_pred, target_names=class_names, digits=4))

print("\nFeature pipeline complete: X_train/X_val/X_test created without leakage.")

### 2. কনফিউশন ম্যাট্রিক্স (Confusion Matrix)

আমরা এখন প্রতিটি ক্লাসিফায়ারের জন্য কনফিউশন ম্যাট্রিক্স তৈরি করব। এটি মডেলের পারফরম্যান্সের একটি বিশদ চিত্র প্রদান করবে।

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

train_counts = train_df['dx'].value_counts().reindex(class_names, fill_value=0)
val_counts = val_df['dx'].value_counts().reindex(class_names, fill_value=0)
test_counts = test_df['dx'].value_counts().reindex(class_names, fill_value=0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, counts, title in [
    (axes[0], train_counts, 'Train Distribution'),
    (axes[1], val_counts, 'Validation Distribution'),
    (axes[2], test_counts, 'Test Distribution'),
]:
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='viridis')
    ax.set_title(title)
    ax.set_xlabel('Disease Type')
    ax.set_ylabel('Number of Samples')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### 3. ROC কার্ভ এবং AUC (ROC Curve and AUC)

ROC কার্ভ এবং AUC (Area Under the Curve) প্রতিটি ক্লাসের জন্য মডেলের পারফরম্যান্স মূল্যায়ন করতে সাহায্য করে। যেহেতু `SVC` ক্লাসিফায়ারে `probability=True` সেট করা আছে, আমরা সরাসরি `predict_proba` ব্যবহার করতে পারি। `LogisticRegression` ও `predict_proba` সাপোর্ট করে।

In [ ]:
from sklearn.metrics import confusion_matrix

plt.figure(figsize=(18, 5))
model_names = list(eval_results.keys())

for i, name in enumerate(model_names):
    pred = eval_results[name]['test_pred']
    cm = confusion_matrix(y_test, pred, labels=list(range(len(class_names))))

    plt.subplot(1, len(model_names), i + 1)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=class_names, yticklabels=class_names
    )
    plt.title(f'{name} (Test)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

plt.tight_layout()
plt.show()

### 4. ফিচার স্পেস ভিজ্যুয়ালাইজেশন (Feature Space Visualization with t-SNE)

t-SNE (t-Distributed Stochastic Neighbor Embedding) ব্যবহার করে 2048-মাত্রিক ফিচার ভেক্টরগুলিকে 2D স্পেসে ভিজ্যুয়ালাইজ করা হবে, যাতে ক্লাসগুলির মধ্যে সেপারাবিলিটি বোঝা যায়।

In [ ]:
from sklearn.metrics import auc, roc_curve
from sklearn.preprocessing import label_binarize

n_classes = len(class_names)
y_test_binarized = label_binarize(y_test, classes=range(n_classes))

plt.figure(figsize=(18, 5))
plot_idx = 1

for name, result in eval_results.items():
    y_score = result['test_proba']
    if y_score is None:
        print(f"Skipping ROC for {name}: predict_proba not available.")
        continue

    plt.subplot(1, len(eval_results), plot_idx)
    plt.plot([0, 1], [0, 1], 'k--', linewidth=1)

    for k in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_binarized[:, k], y_score[:, k])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{class_names[k]} (AUC={roc_auc:.2f})")

    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC: {name}')
    plt.legend(loc='lower right', fontsize=7)
    plot_idx += 1

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import ParameterGrid

search_space = {
    'SVM': {'C': [1.0, 5.0, 10.0], 'gamma': ['scale', 'auto']},
    'KNN': {'n_neighbors': [3, 5, 7]},
    'LR': {'C': [0.5, 1.0, 2.0]}
}

def tune_classifier(name):
    best_cfg = None
    best_model = None
    best_val = -1.0

    for params in ParameterGrid(search_space[name]):
        if name == 'SVM':
            model = SVC(
                kernel='rbf', probability=True, class_weight='balanced',
                random_state=SEED, **params
            )
        elif name == 'KNN':
            model = KNeighborsClassifier(**params)
        else:
            model = LogisticRegression(
                max_iter=2000, class_weight='balanced', random_state=SEED, **params
            )

        model.fit(X_train, y_train)
        val_pred = model.predict(X_val)
        val_acc = accuracy_score(y_val, val_pred)

        if val_acc > best_val:
            best_val = val_acc
            best_cfg = params
            best_model = model

    test_pred = best_model.predict(X_test)
    test_acc = accuracy_score(y_test, test_pred)
    print(f"{name} best params: {best_cfg}")
    print(f"{name} tuned -> Val Acc: {best_val * 100:.2f}% | Test Acc: {test_acc * 100:.2f}%")
    return best_model

tuned_classifiers = {name: tune_classifier(name) for name in ['SVM', 'KNN', 'LR']}

In [ ]:
from sklearn.manifold import TSNE

print("Running t-SNE on test features...")
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_test)
print("t-SNE finished.")

tsne_df = pd.DataFrame({'TSNE1': X_tsne[:, 0], 'TSNE2': X_tsne[:, 1], 'label': y_test})
tsne_df['disease_type'] = tsne_df['label'].map(idx_to_class)

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x='TSNE1', y='TSNE2', hue='disease_type',
    palette=sns.color_palette('tab10', n_classes),
    data=tsne_df, alpha=0.75
)
plt.title('t-SNE of Fused Test Features')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()